In [2]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

print("Project Root:", PROJECT_ROOT)

Project Root: d:\dev\intern\n100


In [3]:
import pandas as pd

from src.etl.loader import ExcelLoader

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

In [ ]:
# Create loaders
core_loader = ExcelLoader(Path("data/raw/core"))
supporting_loader = ExcelLoader(Path("data/raw/supporting"))

# Load datasets
core_datasets = core_loader.load_all()
supporting_datasets = supporting_loader.load_all()

# Merge into one dictionary
datasets = {**core_datasets, **supporting_datasets}

print(f"Core datasets       : {len(core_datasets)}")
print(f"Supporting datasets : {len(supporting_datasets)}")
print(f"Total datasets      : {len(datasets)}")

print("/nLoaded datasets:")
for name in sorted(datasets.keys()):
    print(f"• {name}")

2026-07-20 22:16:35.101 | INFO     | src.etl.loader:discover_excel_files:26 - Discovered 7 Excel files.
2026-07-20 22:16:35.336 | INFO     | src.etl.loader:load_excel:59 - analysis.xlsx loaded successfully (20 rows × 6 columns)
2026-07-20 22:16:35.441 | INFO     | src.etl.loader:load_excel:59 - balancesheet.xlsx loaded successfully (1312 rows × 13 columns)
2026-07-20 22:16:35.533 | INFO     | src.etl.loader:load_excel:59 - cashflow.xlsx loaded successfully (1187 rows × 7 columns)
2026-07-20 22:16:35.557 | INFO     | src.etl.loader:load_excel:59 - companies.xlsx loaded successfully (92 rows × 12 columns)
2026-07-20 22:16:35.614 | INFO     | src.etl.loader:load_excel:59 - documents.xlsx loaded successfully (1585 rows × 4 columns)
2026-07-20 22:16:35.721 | INFO     | src.etl.loader:load_excel:59 - profitandloss.xlsx loaded successfully (1276 rows × 15 columns)
2026-07-20 22:16:35.735 | INFO     | src.etl.loader:load_excel:59 - prosandcons.xlsx loaded successfully (16 rows × 4 columns)
202

Core datasets       : 7
Supporting datasets : 5
Total datasets      : 12

Loaded datasets:
• analysis
• balancesheet
• cashflow
• companies
• documents
• financial_ratios
• market_cap
• peer_groups
• profitandloss
• prosandcons
• sectors
• stock_prices


In [5]:
summary = []

for name, df in datasets.items():

    summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory (KB)": round(df.memory_usage(deep=True).sum()/1024,2),
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum())
    })

summary_df = (
    pd.DataFrame(summary)
    .sort_values("Dataset")
    .reset_index(drop=True)
)

summary_df

,Dataset,Rows,Columns,Memory (KB),Missing Values,Duplicate Rows
0,analysis,20,6,2.80,0,0
1,balancesheet,1312,13,152.32,0,0
2,cashflow,1187,7,82.39,8,0
3,companies,92,12,60.70,9,0
4,documents,1585,4,148.28,52,0
5,financial_ratios,1183,16,165.38,169,0
6,market_cap,551,9,42.64,0,0
7,peer_groups,55,4,2.50,0,0
8,profitandloss,1276,15,167.91,231,0
9,prosandcons,16,4,2.05,6,0


In [ ]:
import pandas as pd

def inspect_dataset(name):
    """
    Display detailed information about a dataset.
    """

    if name not in datasets:
        print(f"Dataset '{name}' not found.")
        return

    df = datasets[name]

    print("=" * 80)
    print(f"DATASET : {name.upper()}")
    print("=" * 80)

    print("Unique IDs:", df["id"].is_unique)
    print("Missing IDs:", df["id"].isna().sum())

    # Shape
    print(f"/nShape : {df.shape[0]} rows × {df.shape[1]} columns")

    # Memory usage
    memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"Memory Usage : {memory:.2f} MB")

    # Data types
    print("/nData Types")
    print("-" * 80)
    print(df.dtypes)

    # Missing values
    print("/nMissing Values")
    print("-" * 80)

    missing = (
    df.isnull()
      .sum()
      .reset_index()
    )

    missing.columns = ["Column", "Missing Values"]

    missing = missing[missing["Missing Values"] > 0]

    print("/nMissing Values")
    print("-" * 80)

    if missing.empty:
        print("No missing values.")
    else:
        display(missing.sort_values("Missing Values", ascending=False))



    # Duplicate rows
    print("/nDuplicate Rows")
    print("-" * 80)
    print(df.duplicated().sum())

    # Columns
    print("/nColumns")
    print("-" * 80)
    print(list(df.columns))

    # First five rows
    print("/nFirst 5 Rows")
    print("-" * 80)
    display(df.head())

    # Summary statistics
    print("/nSummary Statistics")
    print("-" * 80)
    display(df.describe(include="all").T)

In [7]:
inspect_dataset("companies")

DATASET : COMPANIES
Unique IDs: True
Missing IDs: 0

Shape : 92 rows × 12 columns
Memory Usage : 0.06 MB

Data Types
--------------------------------------------------------------------------------
id                     str
company_logo           str
company_name           str
chart_link             str
about_company          str
website                str
nse_profile            str
bse_profile            str
face_value         float64
book_value         float64
roce_percentage    float64
roe_percentage     float64
dtype: object

Missing Values
--------------------------------------------------------------------------------

Missing Values
--------------------------------------------------------------------------------


,Column,Missing Values
11,roe_percentage,2
1,company_logo,1
6,nse_profile,1
5,website,1
7,bse_profile,1
8,face_value,1
9,book_value,1
10,roce_percentage,1



Duplicate Rows
--------------------------------------------------------------------------------
0

Columns
--------------------------------------------------------------------------------
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']

First 5 Rows
--------------------------------------------------------------------------------


,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd\n,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10



Summary Statistics
--------------------------------------------------------------------------------


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,92,92,ABB,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_logo,91,90,https://mkt.in/static/mkt-icons/nifty100/HEROM...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_name,92,92,Abbott India Ltd,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chart_link,92,54,https://in.tradingview.com/chart/qGsydD2w/?sym...,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
about_company,92,92,Abbott India Ltd is one of the leading multina...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
website,91,90,https://www.bajajfinserv.in/,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nse_profile,91,91,https://www.nseindia.com/get-quotes/equity?sym...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bse_profile,91,91,https://www.bseindia.com/stock-share-price/abb...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
face_value,91.0,NaN,NaN,NaN,4.736264,3.974461,1.0,1.0,2.0,10.0,10.0
book_value,91.0,NaN,NaN,NaN,533.098901,1001.931568,24.0,128.0,222.0,464.5,5787.0


In [8]:
inspect_dataset("profitandloss")

DATASET : PROFITANDLOSS
Unique IDs: True
Missing IDs: 0

Shape : 1276 rows × 15 columns
Memory Usage : 0.16 MB

Data Types
--------------------------------------------------------------------------------
id                     int64
company_id               str
year                     str
sales                  int64
expenses               int64
operating_profit     float64
opm_percentage       float64
other_income           int64
interest               int64
depreciation           int64
profit_before_tax      int64
tax_percentage       float64
net_profit             int64
eps                  float64
dividend_payout      float64
dtype: object

Missing Values
--------------------------------------------------------------------------------

Missing Values
--------------------------------------------------------------------------------


,Column,Missing Values
14,dividend_payout,103
11,tax_percentage,95
6,opm_percentage,15
5,operating_profit,13
13,eps,5



Duplicate Rows
--------------------------------------------------------------------------------
0

Columns
--------------------------------------------------------------------------------
['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout']

First 5 Rows
--------------------------------------------------------------------------------


,id,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,profit_before_tax,tax_percentage,net_profit,eps,dividend_payout
0,61,ABB,Dec 2012,1653,1451,202.0,12.0,33,0,19,215,33.0,145,68.0,25.0
1,62,ABB,Mar 2014,2276,2009,267.0,12.0,49,0,22,295,33.0,198,93.0,25.0
2,63,ABB,Mar 2015,2289,1977,312.0,14.0,48,0,15,344,34.0,229,108.0,29.0
3,64,ABB,Mar 2016,2614,2250,365.0,14.0,50,3,14,398,36.0,255,120.0,29.0
4,65,ABB,Mar 2017,2903,2505,398.0,14.0,57,2,16,436,37.0,277,130.0,31.0



Summary Statistics
--------------------------------------------------------------------------------


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,1276.0,NaN,NaN,NaN,724.19279,379.978128,61.0,391.75,736.5,1055.25,1374.0
company_id,1276,100,ADANIPORTS,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,1276,44,TTM,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sales,1276.0,NaN,NaN,NaN,64366.308777,115696.774399,0.0,9830.75,25803.5,68264.75,939838.0
expenses,1276.0,NaN,NaN,NaN,46790.302508,103593.923463,-366.0,4301.75,12069.5,40377.5,870255.0
operating_profit,1263.0,NaN,NaN,NaN,14621.886778,25127.504122,-86697.0,1825.0,5473.0,19312.0,206009.0
opm_percentage,1261.0,NaN,NaN,NaN,349.936558,5819.46656,-57587.0,12.0,21.0,37.0,47971.0
other_income,1276.0,NaN,NaN,NaN,2006.887931,10169.442228,-37775.0,21.0,243.0,1223.0,157189.0
interest,1276.0,NaN,NaN,NaN,6290.561912,20425.108328,-987.0,78.0,514.0,4535.5,284056.0
depreciation,1276.0,NaN,NaN,NaN,2408.515674,5261.460086,0.0,184.75,629.0,2099.5,53226.0


In [9]:
inspect_dataset("balancesheet")

DATASET : BALANCESHEET
Unique IDs: True
Missing IDs: 0

Shape : 1312 rows × 13 columns
Memory Usage : 0.15 MB

Data Types
--------------------------------------------------------------------------------
id                     int64
company_id               str
year                     str
equity_capital       float64
reserves               int64
borrowings             int64
other_liabilities      int64
total_liabilities      int64
fixed_assets           int64
cwip                   int64
investments            int64
other_asset            int64
total_assets           int64
dtype: object

Missing Values
--------------------------------------------------------------------------------

Missing Values
--------------------------------------------------------------------------------
No missing values.

Duplicate Rows
--------------------------------------------------------------------------------
0

Columns
--------------------------------------------------------------------------------
['id

,id,company_id,year,equity_capital,reserves,borrowings,other_liabilities,total_liabilities,fixed_assets,cwip,investments,other_asset,total_assets
0,136,ABB,Dec 2012,21.0,626,0,260,907,109,1,0,798,907
1,137,ABB,Mar 2014,21.0,767,0,351,1139,98,1,0,1040,1139
2,138,ABB,Mar 2015,21.0,916,0,436,1374,96,4,0,1274,1374
3,139,ABB,Mar 2016,21.0,1174,0,421,1616,108,3,0,1505,1616
4,140,ABB,Mar 2017,21.0,1366,0,679,2066,110,6,0,1950,2066



Summary Statistics
--------------------------------------------------------------------------------


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,1312.0,NaN,NaN,NaN,817.480183,384.783676,136.0,481.75,822.5,1150.25,1478.0
company_id,1312,98,PNB,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,1312,54,Mar 2024,99,NaN,NaN,NaN,NaN,NaN,NaN,NaN
equity_capital,1312.0,NaN,NaN,NaN,1158.322942,2275.799533,0.0,94.5,255.0,994.25,13772.0
reserves,1312.0,NaN,NaN,NaN,34160.227896,63254.554119,-2968.0,5260.5,16061.5,41689.75,812687.0
borrowings,1312.0,NaN,NaN,NaN,87109.369665,233130.318826,0.0,379.75,6407.5,63347.25,3107503.0
other_liabilities,1312.0,NaN,NaN,NaN,54232.768293,320733.56972,0.0,2887.0,9200.5,32151.0,5621474.0
total_liabilities,1312.0,NaN,NaN,NaN,176661.21189,435496.93434,0.0,14089.25,56358.0,152215.0,5719121.0
fixed_assets,1312.0,NaN,NaN,NaN,28957.27439,68268.012903,0.0,1216.25,4971.0,23391.5,968747.0
cwip,1312.0,NaN,NaN,NaN,7222.966463,24705.524339,0.0,20.0,288.0,2529.0,338855.0


In [10]:
print(df.duplicated().sum())

0


In [11]:
inspect_dataset("cashflow")

DATASET : CASHFLOW
Unique IDs: True
Missing IDs: 0

Shape : 1187 rows × 7 columns
Memory Usage : 0.08 MB

Data Types
--------------------------------------------------------------------------------
id                      int64
company_id                str
year                      str
operating_activity    float64
investing_activity    float64
financing_activity    float64
net_cash_flow         float64
dtype: object

Missing Values
--------------------------------------------------------------------------------

Missing Values
--------------------------------------------------------------------------------


,Column,Missing Values
3,operating_activity,2
4,investing_activity,2
5,financing_activity,2
6,net_cash_flow,2



Duplicate Rows
--------------------------------------------------------------------------------
0

Columns
--------------------------------------------------------------------------------
['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow']

First 5 Rows
--------------------------------------------------------------------------------


,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0



Summary Statistics
--------------------------------------------------------------------------------


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,1187.0,NaN,NaN,NaN,658.047178,349.094269,37.0,359.5,656.0,961.5,1258.0
company_id,1187,100,TCS,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,1187,51,Mar 2024,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
operating_activity,1185.0,NaN,NaN,NaN,6397.116456,19764.690316,-97820.0,671.0,2711.0,9420.0,158788.0
investing_activity,1185.0,NaN,NaN,NaN,-5229.952743,13720.022026,-144737.0,-5010.0,-1234.0,-271.0,148447.0
financing_activity,1185.0,NaN,NaN,NaN,-174.859916,15282.107821,-256125.0,-2734.0,-491.0,716.0,101904.0
net_cash_flow,1185.0,NaN,NaN,NaN,992.834599,8688.453696,-80593.0,-203.0,46.0,661.0,93392.0


In [13]:
report = pd.read_csv("data/output/validation_failures.csv")
report[report["Rule_ID"] == "DQ-02"].head(10)

,Rule_ID,Severity,Dataset,Row,Message
0,DQ-02,ERROR,profitandloss,47,"Duplicate composite key (ADANIPORTS, Mar 2013)."
1,DQ-02,ERROR,profitandloss,48,"Duplicate composite key (ADANIPORTS, Mar 2014)."
2,DQ-02,ERROR,profitandloss,49,"Duplicate composite key (ADANIPORTS, Mar 2015)."
3,DQ-02,ERROR,profitandloss,50,"Duplicate composite key (ADANIPORTS, Mar 2016)."
4,DQ-02,ERROR,profitandloss,51,"Duplicate composite key (ADANIPORTS, Mar 2017)."
5,DQ-02,ERROR,profitandloss,52,"Duplicate composite key (ADANIPORTS, Mar 2018)."
6,DQ-02,ERROR,profitandloss,53,"Duplicate composite key (ADANIPORTS, Mar 2019)."
7,DQ-02,ERROR,profitandloss,54,"Duplicate composite key (ADANIPORTS, Mar 2020)."
8,DQ-02,ERROR,profitandloss,55,"Duplicate composite key (ADANIPORTS, Mar 2021)."
9,DQ-02,ERROR,profitandloss,56,"Duplicate composite key (ADANIPORTS, Mar 2022)."


In [14]:
report[report["Rule_ID"] == "DQ-03"].head(10)

,Rule_ID,Severity,Dataset,Row,Message
210,DQ-03,ERROR,profitandloss,1164,Company ID 'ULTRACEMCO' not found.
211,DQ-03,ERROR,profitandloss,1165,Company ID 'ULTRACEMCO' not found.
212,DQ-03,ERROR,profitandloss,1166,Company ID 'ULTRACEMCO' not found.
213,DQ-03,ERROR,profitandloss,1167,Company ID 'ULTRACEMCO' not found.
214,DQ-03,ERROR,profitandloss,1168,Company ID 'ULTRACEMCO' not found.
215,DQ-03,ERROR,profitandloss,1169,Company ID 'ULTRACEMCO' not found.
216,DQ-03,ERROR,profitandloss,1170,Company ID 'ULTRACEMCO' not found.
217,DQ-03,ERROR,profitandloss,1171,Company ID 'ULTRACEMCO' not found.
218,DQ-03,ERROR,profitandloss,1172,Company ID 'ULTRACEMCO' not found.
219,DQ-03,ERROR,profitandloss,1173,Company ID 'ULTRACEMCO' not found.


In [22]:
companies = datasets["companies"]

print(companies.columns.tolist())
print(companies.head(10))

['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']
           id                                       company_logo  \
0         ABB   https://mkt.in/static/mkt-icons/nifty100/ABB.png   
1  ADANIENSOL  https://m.economictimes.com/thumb/msid-1173715...   
2    ADANIENT  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
3  ADANIGREEN  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
4  ADANIPORTS  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
5  ADANIPOWER  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
6   AMBUJACEM  https://mkt.in/static/mkt-icons/nifty100/AMBUJ...   
7  APOLLOHOSP  https://mkt.in/static/mkt-icons/nifty100/APOLL...   
8  ASIANPAINT  https://mkt.in/static/mkt-icons/nifty100/ASIAN...   
9        ATGL  https://mkt.in/static/mkt-icons/nifty100/ATGL.png   

                                        company_name  \
0                  

In [24]:
companies[
    companies.astype(str)
             .apply(lambda col: col.str.contains("ADANIPORTS", case=False, na=False))
             .any(axis=1)
]

,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd\n,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.1


In [25]:
duplicates = datasets["profitandloss"][
    datasets["profitandloss"].duplicated(
        subset=["company_id", "year"],
        keep=False
    )
]

print(duplicates.shape)

(26, 15)


In [26]:
duplicates.groupby("company_id").size().sort_values(ascending=False).head(20)

company_id
ADANIPORTS    26
dtype: int64

In [27]:
duplicates[
    duplicates["company_id"] == "ADANIPORTS"
]

,id,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,profit_before_tax,tax_percentage,net_profit,eps,dividend_payout
47,120,ADANIPORTS,Mar 2013,3577,1195,2382.0,67.0,344,542,422,1762,7.0,1639,8.0,12.0
48,121,ADANIPORTS,Mar 2014,4830,1910,2919.0,60.0,685,977,649,1978,12.0,1741,8.0,12.0
49,122,ADANIPORTS,Mar 2015,6152,2250,3902.0,63.0,686,1175,912,2501,7.0,2324,11.0,10.0
50,123,ADANIPORTS,Mar 2016,7109,2532,4577.0,64.0,730,1124,1063,3119,9.0,2856,14.0,8.0
51,124,ADANIPORTS,Mar 2017,8439,3021,5418.0,64.0,1037,1116,1160,4179,7.0,3902,19.0,7.0
52,125,ADANIPORTS,Mar 2018,11323,4166,7157.0,63.0,844,1579,1188,5234,30.0,3690,18.0,11.0
53,126,ADANIPORTS,Mar 2019,10925,4330,6596.0,60.0,1289,1385,1373,5126,21.0,4045,19.0,1.0
54,127,ADANIPORTS,Mar 2020,11873,5926,5947.0,50.0,1928,1951,1680,4244,11.0,3785,19.0,17.0
55,128,ADANIPORTS,Mar 2021,12550,3862,8688.0,69.0,1967,2255,2107,6292,20.0,5049,25.0,20.0
56,129,ADANIPORTS,Mar 2022,17119,7591,9528.0,56.0,1832,2544,3099,5717,13.0,4953,23.0,22.0


In [20]:
companies = datasets["companies"]

print(companies.columns.tolist())

['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']


In [21]:
companies.head()

,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd\n,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10


In [28]:
"ULTRACEMCO" in datasets["companies"]["id"].values

False

In [29]:
duplicates = datasets["profitandloss"][
    datasets["profitandloss"].duplicated(
        subset=["company_id", "year"],
        keep=False
    )
]

duplicates["company_id"].value_counts()

company_id
ADANIPORTS    26
Name: count, dtype: int64

In [30]:
datasets["companies"][
    datasets["companies"]["id"] == "ULTRACEMCO"
]

,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage


In [31]:
companies = set(datasets["companies"]["id"])

for table in ["profitandloss", "balancesheet", "cashflow"]:
    missing = sorted(set(datasets[table]["company_id"]) - companies)
    print(f"{table}: {missing}")

profitandloss: ['ULTRACEMCO', 'UNIONBANK', 'UNITDSPR', 'VBL', 'VEDL', 'WIPRO', 'ZOMATO', 'ZYDUSLIFE']
balancesheet: ['ULTRACEMCO', 'UNIONBANK', 'UNITDSPR', 'VEDL', 'WIPRO', 'ZOMATO', 'ZYDUSLIFE']
cashflow: ['AGTL', 'ULTRACEMCO', 'UNIONBANK', 'UNITDSPR', 'VBL', 'VEDL', 'WIPRO', 'ZOMATO', 'ZYDUSLIFE']


## Sprint 1 – Day 3 (Data Profiling & Validation):

Findings
Verified primary key uniqueness across all datasets.
Identified duplicate business records for ADANIPORTS in the Profit & Loss dataset (26 duplicate (company_id, year) records).
Identified referential integrity violations where financial records reference company IDs absent from the master companies dataset (e.g., ULTRACEMCO).
Generated validation reports summarizing all detected data quality issues.

These are exactly the kinds of findings expected from a data profiling and validation phase.

In [40]:
from src.etl.cleaner import DataCleaner

cleaner = DataCleaner()
cleaned = cleaner.clean(datasets)

print(cleaned["companies"].loc[4, "company_name"])

2026-07-20 22:56:56.719 | INFO     | src.etl.cleaner:clean:20 - Starting data cleaning...
2026-07-20 22:56:56.719 | INFO     | src.etl.cleaner:trim_whitespace:39 - Cleaning Rule C-01: Trimming whitespace...
2026-07-20 22:56:56.722 | INFO     | src.etl.cleaner:trim_whitespace:56 - Whitespace trimming completed.
2026-07-20 22:56:56.722 | INFO     | src.etl.cleaner:clean:30 - Data cleaning completed.


Adani Ports & Special Economic Zone Ltd


In [ ]:
for name, df in cleaned.items():
    print(f"/n{name}")

    for value in ["", "NA", "N/A", "-", "NULL"]:
        count = (df == value).sum().sum()
        if count > 0:
            print(value, count)


analysis

balancesheet

cashflow

companies

documents

profitandloss

prosandcons

financial_ratios

market_cap

peer_groups

sectors

stock_prices


In [ ]:
cleaner = DataCleaner()
cleaned = cleaner.clean(datasets)

for name, df in cleaned.items():
    print(f"/n{name}")
    print(df.dtypes)

2026-07-20 22:57:06.178 | INFO     | src.etl.cleaner:clean:20 - Starting data cleaning...
2026-07-20 22:57:06.179 | INFO     | src.etl.cleaner:trim_whitespace:39 - Cleaning Rule C-01: Trimming whitespace...
2026-07-20 22:57:06.180 | INFO     | src.etl.cleaner:trim_whitespace:56 - Whitespace trimming completed.
2026-07-20 22:57:06.181 | INFO     | src.etl.cleaner:clean:30 - Data cleaning completed.



analysis
id                           int64
company_id                  string
compounded_sales_growth     string
compounded_profit_growth    string
stock_price_cagr            string
roe                         string
dtype: object

balancesheet
id                     int64
company_id            string
year                  string
equity_capital       float64
reserves               int64
borrowings             int64
other_liabilities      int64
total_liabilities      int64
fixed_assets           int64
cwip                   int64
investments            int64
other_asset            int64
total_assets           int64
dtype: object

cashflow
id                      int64
company_id             string
year                   string
operating_activity    float64
investing_activity    float64
financing_activity    float64
net_cash_flow         float64
dtype: object

companies
id                  string
company_logo        string
company_name        string
chart_link          string
about_co

In [51]:
import importlib
import src.etl.cleaner

importlib.reload(src.etl.cleaner)

from src.etl.cleaner import DataCleaner

In [52]:
cleaner = DataCleaner()
cleaned = cleaner.clean(datasets)

2026-07-20 23:02:43.865 | INFO     | src.etl.cleaner:clean:20 - Starting data cleaning...
2026-07-20 23:02:43.868 | INFO     | src.etl.cleaner:trim_whitespace:42 - Cleaning Rule C-01: Trimming whitespace...
2026-07-20 23:02:43.870 | INFO     | src.etl.cleaner:trim_whitespace:59 - Whitespace trimming completed.
2026-07-20 23:02:43.871 | INFO     | src.etl.cleaner:normalize_missing_values:65 - Cleaning Rule C-02: Normalizing missing values...
2026-07-20 23:02:43.885 | INFO     | src.etl.cleaner:normalize_missing_values:81 - Missing value normalization completed.
2026-07-20 23:02:43.885 | INFO     | src.etl.cleaner:convert_numeric_columns:87 - Cleaning Rule C-03: Converting numeric columns...
2026-07-20 23:02:43.893 | INFO     | src.etl.cleaner:convert_numeric_columns:139 - Numeric conversion completed.
2026-07-20 23:02:43.893 | INFO     | src.etl.cleaner:remove_duplicate_business_records:146 - Cleaning Rule C-04: Removing duplicate business records...
2026-07-20 23:02:43.897 | INFO     |

In [53]:
cleaned["profitandloss"].duplicated(
    subset=["company_id", "year"]
).sum()

np.int64(0)

In [54]:
print(datasets["profitandloss"].shape)
print(cleaned["profitandloss"].shape)

(1276, 15)
(1263, 15)


In [55]:
cleaner.export_cleaned_data()

2026-07-20 23:03:42.735 | INFO     | src.etl.cleaner:export_cleaned_data:186 - Cleaned datasets exported to data\cleaned


In [3]:
import pandas as pd

df = pd.read_csv("E:/dev1/dev/intern/n100/data/cleaned/balancesheet.csv")
print(df.columns.tolist())

['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets']
